# LIVR-Mini-Benchmark — Evaluation Protocol trên Kaggle

Notebook này chạy một thí nghiệm đánh giá có kiểm soát cho **LIVR (Latent Implicit Visual Reasoning)** trên GPU Kaggle T4. Mục tiêu không phải tái tạo đầy đủ toàn bộ paper, mà là kiểm tra trong điều kiện mini-benchmark xem cơ chế **latent bottleneck** có tạo thêm lợi ích so với fine-tuning thông thường hay không.

Thiết kế thí nghiệm dùng cùng một mô hình nền, cùng dữ liệu train/test và cùng ngân sách epoch cho ba mốc so sánh:

| Mốc | Huấn luyện | Latent Tokens | Bottleneck Mask | Vai trò |
| :--- | :--- | :---: | :---: | :--- |
| Zero-shot | Không train | Không | Không | Năng lực sẵn có của base model |
| Direct SFT | 5 epoch | Không | Không | Baseline fine-tuning chuẩn |
| LIVR | 2 epoch Stage 1 + 3 epoch Stage 2 | Có, `K=16` | Có ở Stage 1 | Phương pháp cần kiểm chứng |

Protocol cố định: **500 mẫu train mỗi dataset**, **200 mẫu test mỗi dataset**, chọn bằng split stratified deterministic theo metadata của từng dataset, train trên `MathVista + CV-Bench`, rồi đánh giá riêng từng dataset. Notebook chỉ dùng kết quả sinh ra trong lần chạy hiện tại, không dùng checkpoint Direct SFT bên ngoài và không dùng số fallback hardcode.

## 1. Thiết Lập Môi Trường

Cell đầu tiên chuẩn bị môi trường Kaggle để notebook có thể chạy tự động từ đầu đến cuối.

Các việc chính:

- Đọc `HF_TOKEN` từ Kaggle Secrets để tải model và dataset trên Hugging Face.
- Clone hoặc cập nhật repo từ nhánh `develop`.
- Đưa working directory về thư mục repo để import được các module trong `src/`.

Sau cell này, toàn bộ code được dùng là code trong repo vừa clone/pull, vì vậy cần chắc chắn nhánh `develop` đã chứa phiên bản mới nhất của `src/model.py`, `src/mask_kaggle.py` và notebook này.

In [1]:
# =========================================================================
# CELL 1: KHAI BÁO HF_TOKEN VÀ ĐỒNG BỘ CODE TỪ GITHUB TRÊN KAGGLE
# =========================================================================
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("➔ Đã nạp HF_TOKEN thành công từ Kaggle Secrets!")
except Exception as e:
    print(f"➔ Không nạp được secrets: {e}. Vui lòng tự gán os.environ['HF_TOKEN'] nếu cần.")

REPO_URL = "https://github.com/dinhtri445/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /kaggle/working
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

➔ Đã nạp HF_TOKEN thành công từ Kaggle Secrets!
/kaggle/working
---> Đang thực hiện clone repo https://github.com/dinhtri445/LIVR-Mini-Benchmark.git (nhánh develop)...
Cloning into 'LIVR-Mini-Benchmark'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (251/251), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 251 (delta 157), reused 164 (delta 76), pack-reused 0 (from 0)
Receiving objects: 100% (251/251), 10.80 MiB | 33.61 MiB/s, done.
Resolving deltas: 100% (157/157), done.
/kaggle/working/LIVR-Mini-Benchmark


## 2. Cài Đặt Thư Viện và Kiểm Tra GPU

Cell này cài dependencies từ `requirements.txt` và in thông tin CUDA/GPU.

Điểm quan trọng cho Kaggle:

- `transformers` được pin về `4.51.3` để tránh thay đổi API generation ở các bản mới hơn làm Qwen2.5-VL crash khi gọi `generate()`.
- Pipeline dùng QLoRA 4-bit qua `bitsandbytes` để Qwen2.5-VL-3B có thể train trên T4 16GB.
- T4 không tối ưu cho `bfloat16`, nên forward/eval dùng `float16` autocast. Các tham số trainable LoRA/latent embedding được giữ ở `float32` để `GradScaler` không lỗi FP16 gradients, rồi dùng gradient clipping.

Nếu cell này báo GPU không khả dụng, không nên chạy các cell training phía sau.

In [2]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & PHÁT HIỆN GPU
# =========================================================================
# Cài đặt các thư viện lõi từ requirements.txt
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 74.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 102.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 55.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.10.1
    Uninstalling huggingface_hub-1.10.1:
      Successfully uninstalled huggingface_hub-1.10.1
  Attempting uninst

## 3. Cấu Hình Evaluation

Cell này đọc `config/evaluation_config.json` rồi override các tham số dành riêng cho Kaggle. Đây là nơi định nghĩa protocol cuối cùng của thí nghiệm.

| Tham số | Giá trị | Ý nghĩa |
| :--- | :---: | :--- |
| `base_model_id` | `Qwen/Qwen2.5-VL-3B-Instruct` | Backbone dùng cho cả ba mốc so sánh |
| `K` | 16 | Số latent tokens của LIVR |
| `train_samples` | 500 / dataset | Mẫu đầu mỗi dataset dùng để train Direct SFT và LIVR |
| `test_samples` | 200 / dataset | Mẫu cuối mỗi dataset dùng để đánh giá |
| `fine_tune_epochs` | 5 | Direct SFT train 5 epoch |
| `livr_stage1_epochs` | 2 | LIVR bottleneck training |
| `livr_stage2_epochs` | 3 | LIVR standard-mask fine-tuning |
| `grad_accumulation_steps` | 8 | Mô phỏng batch lớn hơn trên VRAM T4 |

Việc giữ tổng epoch của Direct SFT và LIVR bằng nhau giúp so sánh tập trung vào đóng góp của latent bottleneck thay vì khác biệt compute.

In [3]:
# =========================================================================
# CELL 3: NẠP FILE CẤU HÌNH ĐÁNH GIÁ VÀ OVERRIDE CHO KAGGLE
# =========================================================================
import json

with open("config/evaluation_config.json", "r", encoding="utf-8") as f:
    eval_config = json.load(f)

# Cấu hình lại đường dẫn lưu trữ sang thư mục Kaggle
eval_config["checkpoint_path"] = "/kaggle/input/notebooks/nguyentien15/livr-mini-benchmark/checkpoints/livr_mini_checkpoint.pt"
eval_config["output_dir"] = "/kaggle/working/checkpoints/evaluation"

if "visu_logic" in eval_config["eval_datasets"]:
    eval_config["eval_datasets"]["math_vista"] = eval_config["eval_datasets"].pop("visu_logic")
    eval_config["eval_datasets"]["math_vista"]["name"] = "MathVista"
    eval_config["eval_datasets"]["math_vista"]["huggingface_path"] = "AI4Math/MathVista"

# Protocol final: 500 train + 200 eval cho mỗi dataset.
for ds_key in ["math_vista", "cv_bench"]:
    eval_config["eval_datasets"][ds_key]["train_samples"] = 500
    eval_config["eval_datasets"][ds_key]["test_samples"] = 200

eval_config["fine_tune_epochs"] = 5
eval_config["livr_stage1_epochs"] = 2
eval_config["livr_stage2_epochs"] = 3
eval_config["learning_rate"] = eval_config.get("learning_rate", 5e-5)
eval_config["grad_accumulation_steps"] = eval_config.get("grad_accumulation_steps", 8)

print("KAGGLE EVALUATION CONFIGURATION:")
print(json.dumps(eval_config, indent=2))

KAGGLE EVALUATION CONFIGURATION:
{
  "base_model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
  "checkpoint_path": "/kaggle/input/notebooks/nguyentien15/livr-mini-benchmark/checkpoints/livr_mini_checkpoint.pt",
  "K": 16,
  "eval_datasets": {
    "math_vista": {
      "name": "MathVista",
      "huggingface_path": "AI4Math/MathVista",
      "train_samples": 500,
      "val_samples": 100,
      "test_samples": 200
    },
    "cv_bench": {
      "name": "CV-Bench",
      "huggingface_path": "nyu-visionx/CV-Bench",
      "train_samples": 500,
      "val_samples": 100,
      "test_samples": 200
    }
  },
  "fine_tune_epochs": 5,
  "livr_stage1_epochs": 2,
  "livr_stage2_epochs": 3,
  "learning_rate": 5e-05,
  "batch_size_per_device": 1,
  "grad_accumulation_steps": 8,
  "output_dir": "/kaggle/working/checkpoints/evaluation",
  "_comment": "Train 500 mau/dataset (1000 combined). Direct SFT: 5 epochs. LIVR: Stage1=2 epochs (40%) + Stage2=3 epochs (60%) = 5 epochs total. Ty le 2:3 theo paper goc (4:6

## 4. Tải Dataset Đánh Giá

Notebook dùng hai dataset public có ảnh inline để phù hợp môi trường Kaggle:

| Dataset | Split dùng | Vai trò trong thí nghiệm |
| :--- | :--- | :--- |
| CV-Bench | `test` | Spatial / depth / object-centric visual reasoning |
| MathVista | `testmini` | Visual mathematical reasoning, chart/diagram/table understanding |

Đối chiếu với dataset card:

- **CV-Bench** có sẵn `prompt` đã format question + choices, `answer` là option label như `(A)`, `(B)`, và metadata `type/source/task`.
- **MathVista** có `query` chứa hint chính thức, `question_type`, `answer_type`, `choices`; với multi-choice, label public trong cột `answer` là text option, còn `query` thường yêu cầu trả lời option letter.

Vì CV-Bench được sắp theo cụm source/task, notebook không còn lấy 500 mẫu đầu và 200 mẫu cuối một cách tuần tự. Thay vào đó, cell helper tạo split **stratified deterministic**: CV-Bench stratify theo `type/source/task`, MathVista stratify theo `question_type/answer_type`. Đây vẫn là **domain-adapted mini evaluation**, không phải protocol leaderboard chính thức của các dataset.

In [4]:
# =========================================================================
# CELL 4: TẢI NOVEL DATASETS TRÊN KAGGLE (MATHVISTA & CV-BENCH)
# =========================================================================
from datasets import load_dataset
import os

# Lưu cache dataset vào thư mục làm việc của Kaggle
cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)
print(f"-> Sử dụng thư mục lưu cache dataset: {cache_dir}")

print("---> Đang tải tập dữ liệu MathVista (split testmini có ảnh)... ")
try:
    # Tải split testmini (1,000 mẫu) của MathVista để đánh giá nhanh và thích nghi
    mathvista_dataset = load_dataset("AI4Math/MathVista", split="testmini", cache_dir=cache_dir)
    print("MathVista testmini Dataset:", mathvista_dataset)
except Exception as e:
    print(f"Lỗi tải MathVista: {e}.")
    mathvista_dataset = None

print("\n---> Đang tải tập dữ liệu CV-Bench...")
try:
    cv_dataset = load_dataset("nyu-visionx/CV-Bench", cache_dir=cache_dir)
    print("CV-Bench Dataset:", cv_dataset)
except Exception as e:
    print(f"Lỗi tải CV-Bench: {e}.")
    cv_dataset = None

-> Sử dụng thư mục lưu cache dataset: /kaggle/working/dataset_cache
---> Đang tải tập dữ liệu MathVista (split testmini có ảnh)... 


README.md: 0.00B [00:00, ?B/s]

data/testmini-00000-of-00001-725687bf7a1(…):   0%|          | 0.00/142M [00:00<?, ?B/s]

data/test-00000-of-00002-6b81bd7f7e2065e(…):   0%|          | 0.00/358M [00:00<?, ?B/s]

data/test-00001-of-00002-6a611c71596db30(…):   0%|          | 0.00/386M [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

test_2d.parquet:   0%|          | 0.00/185M [00:00<?, ?B/s]

test_3d.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2638 [00:00<?, ? examples/s]

CV-Bench Dataset: DatasetDict({
    test: Dataset({
        features: ['idx', 'type', 'task', 'image', 'question', 'choices', 'answer', 'prompt', 'filename', 'source', 'source_dataset', 'source_filename', 'target_class', 'target_size', 'bbox'],
        num_rows: 2638
    })
})


## 5. Khởi Tạo Model và Snapshot Ban Đầu

Cell này tạo một model Qwen2.5-VL-3B duy nhất, sau đó gắn LoRA và thêm `K=16` latent tokens. Để tiết kiệm VRAM, notebook không tạo ba model riêng biệt cho Zero-shot, Direct SFT và LIVR.

Thay vào đó, sau khi khởi tạo xong, notebook lưu snapshot gồm:

- Toàn bộ LoRA tensors ban đầu.
- 16 hàng embedding tương ứng với latent tokens.

Trước mỗi nhánh thí nghiệm, notebook restore từ snapshot này. Cách này quan trọng vì **không được zero LoRA thủ công** sau Direct SFT: nếu zero cả LoRA A/B, gradient có thể bị nghẽn và loss LIVR sẽ đứng im qua epoch.

In [5]:
# =========================================================================
# CELL 5: LOAD BASE MODEL, KHỞI TẠO LORA/LATENTS, SNAPSHOT STATE BAN ĐẦU
# =========================================================================
import os
import gc
import copy
import json
import random
import torch
import numpy as np
from PIL import Image
from src.model import LIVRModelManager
from src.mask_kaggle import patch_model_for_livr

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = eval_config["checkpoint_path"]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

manager = LIVRModelManager(
    model_id=eval_config["base_model_id"],
    K=eval_config["K"],
    device=device,
    load_in_4bit=True,
)
model = manager.setup_peft_and_freezing()
processor = manager.processor

def ensure_trainable_params_fp32(model):
    dtype_counts = {}
    converted = 0
    for name, param in model.named_parameters():
        if param.requires_grad:
            dtype_counts[str(param.dtype)] = dtype_counts.get(str(param.dtype), 0) + param.numel()
            if param.dtype != torch.float32:
                param.data = param.data.float()
                converted += 1
    print(f"---> Trainable dtype before/after guard: {dtype_counts}; converted_tensors={converted}")

ensure_trainable_params_fp32(model)

LOAD_IMPLEMENT_CHECKPOINT = False
if LOAD_IMPLEMENT_CHECKPOINT and os.path.exists(checkpoint_path):
    print(f"---> Đang khôi phục trọng số huấn luyện từ: {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    missing, unexpected = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    print(f"---> load_state_dict missing={len(missing)} unexpected={len(unexpected)}")
    with torch.no_grad():
        model.get_input_embeddings().weight[manager.latent_token_ids] = checkpoint["latent_embeddings"].to(device)
    print("---> Đã khôi phục checkpoint implement.")
else:
    print("---> Protocol hiện tại train Direct SFT và LIVR từ cùng base pretrained online.")

patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id,
)

def is_lora_param(name):
    return "lora_" in name.lower()

initial_trainable_state = {
    n: p.detach().clone().cpu()
    for n, p in model.named_parameters()
    if p.requires_grad and is_lora_param(n)
}
initial_latent_embeddings = model.get_input_embeddings().weight[manager.latent_token_ids].detach().clone().cpu()
print(f"---> Snapshot LoRA state: {len(initial_trainable_state)} tensors + {len(manager.latent_token_ids)} latent rows.")

def restore_initial_trainable_state(model, state):
    """Restore LoRA + 16 latent embedding rows without reloading the full 4-bit base model."""
    with torch.no_grad():
        restored = 0
        for name, param in model.named_parameters():
            if name in state:
                param.copy_(state[name].to(device=param.device, dtype=param.dtype))
                restored += 1
        embed_weight = model.get_input_embeddings().weight
        embed_weight[manager.latent_token_ids] = initial_latent_embeddings.to(device=embed_weight.device, dtype=embed_weight.dtype)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"---> Restored {restored}/{len(state)} LoRA tensors + latent rows to initial state.")

def extract_lora_state_dict(model):
    return {
        k: v.detach().cpu()
        for k, v in model.state_dict().items()
        if "lora_" in k.lower()
    }

print("---> Model ready. Direct SFT và LIVR sẽ dùng cùng initial_trainable_state.")

2026-06-20 14:09:08.686022: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781964549.074196      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781964549.195420      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781964550.200729      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781964550.200764      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781964550.200767      58 computation_placer.cc:177] computation placer alr

Loading processor & tokenizer for Qwen/Qwen2.5-VL-3B-Instruct...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Resizing token embeddings to 151681...
Configuring PEFT LoRA...
Freezing base parameters & setup embedding hooks...
---> Tham số có thể huấn luyện: 347,795,456 / 2,070,654,976 (16.80%)
---> Trainable dtype before/after guard: {'torch.float32': 347795456}; converted_tensors=0
---> Protocol hiện tại train Direct SFT và LIVR từ cùng base pretrained online.
---> Đã tích hợp Custom Attention Mask tương thích Kaggle thành công!
---> Snapshot LoRA state: 696 tensors + 16 latent rows.
---> Model ready. Direct SFT và LIVR sẽ dùng cùng initial_trainable_state.


## 5b-I. Helper Chung và Zero-shot Baseline

Cell này định nghĩa toàn bộ helper dùng chung cho ba mốc so sánh, để tránh mỗi baseline có prompt format hoặc answer matching khác nhau.

Các helper chính:

- Chuẩn hóa item dataset thành chat conversation của Qwen2.5-VL, ưu tiên `prompt` của CV-Bench và `query` của MathVista.
- Thêm task prefix theo dataset: CV-Bench trả lời option letter; MathVista trả lời option letter cho multi-choice và đáp án cuối ngắn cho free-form.
- Tạo split train/test cố định 500/200 bằng stratified deterministic split.
- Chuẩn bị input tensor cho hai chế độ: không latent tokens và có latent tokens.
- Train loop chung với AMP, GradScaler, gradient clipping và OOM guard.
- Evaluation loop chung với `smart_match_answer` thiên về đáp án trực tiếp/cuối cùng; với multiple-choice dùng thêm answer aliases để chấp nhận cả option letter và text option hợp lệ.

Cuối cell này chạy Zero-shot trên test split. Zero-shot không chèn latent tokens và không fine-tune, nên đây là mốc thấp nhất/điểm tham chiếu năng lực sẵn có của base model.

In [6]:
# =========================================================================
# CELL 5b-I: HELPER CHUNG + ZERO-SHOT BASELINE
# =========================================================================
import os
import re
import gc
import json
import copy
import random
from collections import defaultdict, Counter
from PIL import Image
from datasets import DatasetDict
import torch
from tqdm import tqdm

cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(eval_config["output_dir"], exist_ok=True)

def _normalize_answer_text(text):
    text = re.sub(r"\s+", " ", str(text).strip())
    return text.strip(" .,:;\t\n\r")

def _extract_numbers(text):
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))]

def _choice_letter(index):
    return chr(65 + index)

def _choice_label(index):
    return f"({_choice_letter(index)})"

def _extract_choice_answer(text):
    clean = _normalize_answer_text(text).upper()
    compact = clean.replace("(", "").replace(")", "")
    if re.fullmatch(r"[A-Z]", compact):
        return compact.lower()

    patterns = [
        r"^\(?([A-Z])\)?[\.\):]",
        r"(?:FINAL\s+ANSWER|ANSWER|OPTION|CHOICE)\s*(?:IS|:)?\s*\(?([A-Z])\)?",
        r"\(([A-Z])\)\s*$",
        r"\b([A-Z])\b\s*$",
    ]
    for pattern in patterns:
        match = re.search(pattern, clean)
        if match:
            return match.group(1).lower()
    return None

def _choice_index_from_answer(answer, choices):
    if not choices:
        return None
    letter = _extract_choice_answer(answer)
    if letter:
        idx = ord(letter.upper()) - ord("A")
        if 0 <= idx < len(choices):
            return idx

    target = _normalize_answer_text(answer).lower()
    for idx, choice in enumerate(choices):
        if _normalize_answer_text(choice).lower() == target:
            return idx
    return None

def canonical_answer_and_aliases(item, dataset_name=""):
    raw_answer = str(item.get("answer", item.get("label", ""))).strip()
    choices = item.get("choices", None)
    aliases = [raw_answer]

    if choices and isinstance(choices, list):
        choice_idx = _choice_index_from_answer(raw_answer, choices)
        if choice_idx is not None:
            letter = _choice_letter(choice_idx)
            label = _choice_label(choice_idx)
            choice_text = str(choices[choice_idx]).strip()
            canonical = label
            aliases.extend([
                label,
                letter,
                choice_text,
                f"{label} {choice_text}",
                f"{letter}. {choice_text}",
            ])
            deduped = list(dict.fromkeys([a for a in aliases if str(a).strip()]))
            return canonical, deduped, raw_answer

    deduped = list(dict.fromkeys([a for a in aliases if str(a).strip()]))
    return raw_answer, deduped, raw_answer

def _numeric_answer_candidates(text):
    clean = str(text).replace(",", "")
    cue_candidates = []
    cue_patterns = [
        r"(?:final answer|answer)\s*(?:is|:)?\s*([^\n]*)",
        r"(?:therefore|thus|so)\s*,?\s*([^\n]*)",
        r"=\s*\$?\s*(-?\d+(?:\.\d+)?)",
    ]
    for pattern in cue_patterns:
        for match in re.finditer(pattern, clean, flags=re.IGNORECASE):
            nums = _extract_numbers(match.group(1))
            if nums:
                cue_candidates.extend(nums)
    if cue_candidates:
        return [cue_candidates[-1]]

    nums = _extract_numbers(clean)
    if not nums:
        return []
    if len({round(num, 8) for num in nums}) == 1:
        return [nums[0]]
    return []

def _match_single_answer(pred, target):
    if pred is None or target is None:
        return False
    pred = str(pred).strip()
    target = str(target).strip()
    if not pred or not target:
        return False

    if _normalize_answer_text(pred).lower() == _normalize_answer_text(target).lower():
        return True

    p = _normalize_answer_text(pred).lower().replace("(", "").replace(")", "")
    t = _normalize_answer_text(target).lower().replace("(", "").replace(")", "")
    if p == t:
        return True

    if len(t) == 1 and t.isalpha():
        return _extract_choice_answer(pred) == t

    target_nums = _extract_numbers(target)
    if target_nums:
        pred_nums = _numeric_answer_candidates(pred)
        return any(abs(a - b) < 1e-6 for a in pred_nums for b in target_nums)

    answer_match = re.search(r"(?:final answer|answer)\s*(?:is|:)?\s*(.+)$", pred, flags=re.IGNORECASE)
    if answer_match:
        return _normalize_answer_text(answer_match.group(1)).lower() == _normalize_answer_text(target).lower()
    return False

def smart_match_answer(pred, target, aliases=None):
    candidates = [target]
    if aliases:
        candidates.extend(aliases)
    candidates = list(dict.fromkeys([str(c).strip() for c in candidates if str(c).strip()]))
    return any(_match_single_answer(pred, candidate) for candidate in candidates)

def get_primary_split(raw_dataset):
    if raw_dataset is None:
        return None
    if isinstance(raw_dataset, DatasetDict):
        if "test" in raw_dataset:
            return raw_dataset["test"]
        return raw_dataset[list(raw_dataset.keys())[0]]
    return raw_dataset

def load_image_from_item(item):
    image = item.get("decoded_image") or item.get("image")
    if isinstance(image, str) and image:
        img_path = os.path.join(cache_dir, image)
        if os.path.exists(img_path):
            image = Image.open(img_path).convert("RGB")
        else:
            image = None
    elif isinstance(image, Image.Image):
        image = image.convert("RGB")
    return image

def task_prefix_for_dataset(dataset_name):
    dataset_name = (dataset_name or "").lower()
    if "cv" in dataset_name:
        return "[Task: Spatial Choice] Answer with the option letter only, e.g. (A) or (B). Do not explain.\n"
    if "math" in dataset_name:
        return "[Task: Math Reasoning] For multiple-choice questions, answer with the option letter only. For free-form questions, answer with the final exact value or a very short phrase only. Do not explain.\n"
    return ""

def _first_nonempty(*values):
    for value in values:
        if value is not None and str(value).strip():
            return str(value)
    return ""

def format_prompt_answer(item, dataset_name=""):
    dataset_name_l = (dataset_name or "").lower()
    if "math" in dataset_name_l:
        prompt = _first_nonempty(item.get("query"), item.get("prompt"), item.get("question"))
    else:
        prompt = _first_nonempty(item.get("prompt"), item.get("query"), item.get("question"))

    answer, answer_aliases, original_answer = canonical_answer_and_aliases(item, dataset_name=dataset_name)
    choices = item.get("choices", None)
    if choices and isinstance(choices, list):
        if "(A)" not in prompt:
            opts = " ".join([f"({chr(65+i)}) {opt}" for i, opt in enumerate(choices)])
            prompt = f"{prompt}\nSelect from the following choices:\n{opts}"
        if "letter" not in prompt.lower():
            prompt = f"{prompt}\nAnswer with the option letter directly."

    prefix = task_prefix_for_dataset(dataset_name)
    if prefix and not str(prompt).lstrip().startswith("[Task:"):
        prompt = prefix + str(prompt).strip()
    return prompt, answer, answer_aliases, original_answer

def item_to_conversation(item, dataset_name=""):
    image = load_image_from_item(item)
    prompt, answer, answer_aliases, original_answer = format_prompt_answer(item, dataset_name=dataset_name)
    user_content = []
    if image is not None:
        user_content.append({"type": "image", "image": image})
    user_content.append({"type": "text", "text": prompt})
    return {
        "conversation": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]},
        ],
        "prompt": prompt,
        "answer": answer,
        "answer_aliases": answer_aliases,
        "original_answer": original_answer,
    }

def stratify_key_for_item(item, dataset_name=""):
    dataset_name_l = (dataset_name or "").lower()
    if "cv" in dataset_name_l:
        return "|".join([
            str(item.get("type", "unknown")),
            str(item.get("source", "unknown")),
            str(item.get("task", "unknown")),
        ])
    if "math" in dataset_name_l:
        return "|".join([
            str(item.get("question_type", "unknown")),
            str(item.get("answer_type", "unknown")),
        ])
    return "all"

def _allocate_proportional_counts(group_sizes, total):
    total_available = sum(group_sizes.values())
    total = min(total, total_available)
    if total <= 0 or total_available <= 0:
        return {k: 0 for k in group_sizes}

    raw = {k: (v / total_available) * total for k, v in group_sizes.items()}
    counts = {k: min(group_sizes[k], int(raw[k])) for k in group_sizes}
    remaining = total - sum(counts.values())
    ranked = sorted(group_sizes, key=lambda k: (raw[k] - int(raw[k]), group_sizes[k]), reverse=True)
    while remaining > 0:
        progressed = False
        for key in ranked:
            if counts[key] < group_sizes[key]:
                counts[key] += 1
                remaining -= 1
                progressed = True
                if remaining == 0:
                    break
        if not progressed:
            break
    return counts

def make_stratified_train_test_indices(raw_dataset, train_samples, test_samples, dataset_name="", seed=42):
    ds = get_primary_split(raw_dataset)
    if ds is None:
        return {"train": [], "test": []}

    grouped = defaultdict(list)
    for idx, item in enumerate(ds):
        grouped[stratify_key_for_item(item, dataset_name)].append(idx)

    rng = random.Random(seed)
    for indices in grouped.values():
        rng.shuffle(indices)

    group_sizes = {key: len(indices) for key, indices in grouped.items()}
    test_counts = _allocate_proportional_counts(group_sizes, test_samples)
    remaining_sizes = {key: group_sizes[key] - test_counts[key] for key in group_sizes}
    train_counts = _allocate_proportional_counts(remaining_sizes, train_samples)

    train_indices, test_indices = [], []
    for key, indices in grouped.items():
        n_test = test_counts[key]
        n_train = train_counts[key]
        test_indices.extend(indices[:n_test])
        train_indices.extend(indices[n_test:n_test + n_train])

    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return {"train": train_indices, "test": test_indices}

def print_split_audit(raw_dataset, split_indices, dataset_name, title):
    ds = get_primary_split(raw_dataset)
    if ds is None:
        return
    print(f"\n{title} split audit:")
    for role in ["train", "test"]:
        indices = split_indices.get(role, [])
        keys = Counter(stratify_key_for_item(ds[int(idx)], dataset_name) for idx in indices)
        preview = dict(keys.most_common(8))
        print(f"  {role}: n={len(indices)} | strata={preview}")

def build_formatted_split(raw_dataset, num_samples, split_role, dataset_name="", indices=None):
    ds = get_primary_split(raw_dataset)
    if ds is None:
        return []
    if indices is None:
        if split_role == "train":
            indices = range(0, min(num_samples, len(ds)))
        elif split_role == "test":
            start = max(0, len(ds) - num_samples)
            indices = range(start, len(ds))
        else:
            raise ValueError(f"Unknown split_role={split_role}")
    else:
        indices = list(indices)[:num_samples]
    return [item_to_conversation(ds[int(idx)], dataset_name=dataset_name) for idx in indices]

def prepare_vqa_inputs(processor, conversation, latent_tokens=None, device="cuda", include_labels=True):
    conv = copy.deepcopy(conversation)
    if latent_tokens:
        latent_str = "".join(latent_tokens)
        for msg in conv:
            if msg["role"] == "user":
                for content_item in msg["content"]:
                    if content_item["type"] == "text":
                        content_item["text"] = f"{content_item['text'].strip()}\n{latent_str}"

    is_training = include_labels and conv[-1]["role"] == "assistant"
    text_conv = conv if is_training else [m for m in conv if m["role"] == "user"]
    full_text = processor.apply_chat_template(text_conv, tokenize=False, add_generation_prompt=not is_training)
    user_conv = [m for m in conv if m["role"] == "user"]
    prompt_text = processor.apply_chat_template(user_conv, tokenize=False, add_generation_prompt=True)

    images = []
    for msg in conv:
        if msg["role"] == "user":
            for content_item in msg["content"]:
                if content_item["type"] == "image" and content_item["image"] is not None:
                    image = content_item["image"]
                    if isinstance(image, str):
                        image = Image.open(image).convert("RGB")
                    images.append(image)

    images_arg = [images] if images else None
    full_inputs = processor(text=[full_text], images=images_arg, padding=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in full_inputs.items()}

    if include_labels:
        prompt_inputs = processor(text=[prompt_text], images=images_arg, padding=True, return_tensors="pt")
        labels = inputs["input_ids"].clone()
        labels[:, :prompt_inputs["input_ids"].size(1)] = -100
        inputs["labels"] = labels
    return inputs

def train_model_on_data(model, train_data, *, latent_tokens=None, epochs=5, stage1_epochs=0, run_name="SFT"):
    model.train()
    lr = eval_config.get("learning_rate", 5e-5)
    grad_accum_steps = eval_config.get("grad_accumulation_steps", 8)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scaler = torch.amp.GradScaler("cuda")
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    loss_history = []

    for epoch in range(1, epochs + 1):
        current_stage = 1 if epoch <= stage1_epochs else 2
        model.livr_stage = current_stage
        epoch_loss = 0.0
        seen = 0
        optimizer.zero_grad(set_to_none=True)
        progress_bar = tqdm(train_data, desc=f"{run_name} [Stage {current_stage}] Epoch {epoch}/{epochs}")

        for step, sample in enumerate(progress_bar):
            try:
                inputs = prepare_vqa_inputs(
                    processor=processor,
                    conversation=sample["conversation"],
                    latent_tokens=latent_tokens,
                    device="cuda",
                    include_labels=True,
                )
                with torch.amp.autocast("cuda", dtype=torch.float16):
                    outputs = model(**inputs)
                    loss = outputs.loss / grad_accum_steps

                if torch.isnan(loss.detach()) or torch.isinf(loss.detach()):
                    print(f"\n[WARNING] {run_name}: NaN/Inf loss tại step {step}, bỏ qua batch.")
                    optimizer.zero_grad(set_to_none=True)
                    del inputs, outputs, loss
                    continue

                scaler.scale(loss).backward()
                batch_loss = loss.item() * grad_accum_steps
                epoch_loss += batch_loss
                seen += 1

                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_data):
                    fp16_grad_names = [name for name, param in model.named_parameters() if param.requires_grad and param.grad is not None and param.grad.dtype == torch.float16]
                    if fp16_grad_names:
                        raise RuntimeError(f"{run_name}: FP16 gradients remain before scaler.unscale_: {fp16_grad_names[:5]}. Re-run Cell 5 so trainable LoRA/latent params are cast to float32.")
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                progress_bar.set_postfix({"Loss": f"{batch_loss:.4f}", "Scale": f"{scaler.get_scale():.0f}"})
                del inputs, outputs, loss
                if step % 10 == 0:
                    gc.collect()
                    torch.cuda.empty_cache()
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"\n[WARNING] {run_name}: OOM tại step {step}, dọn CUDA cache và bỏ qua batch.")
                    optimizer.zero_grad(set_to_none=True)
                    del e
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                raise

        avg_loss = epoch_loss / max(seen, 1)
        loss_history.append(avg_loss)
        print(f"➔ {run_name} Epoch {epoch}/{epochs} - Stage {current_stage} - Average Loss: {avg_loss:.4f} - Scaler Scale: {scaler.get_scale()}")

    return loss_history

def evaluate_formatted_data(model, data_list, *, latent_tokens=None, name="Eval", max_samples=200, apply_stage1=False):
    model.eval()
    correct = 0
    total = 0
    log_entries = []
    model.livr_stage = 1 if apply_stage1 else 2

    print(f"\n➔ Đang chạy đánh giá trên {name} ({min(max_samples, len(data_list))} mẫu)...")
    with torch.no_grad():
        for i, sample in enumerate(data_list[:max_samples]):
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=sample["conversation"],
                latent_tokens=latent_tokens,
                device="cuda",
                include_labels=False,
            )
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=32,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.pad_token_id,
                    eos_token_id=processor.tokenizer.eos_token_id,
                )
            input_len = inputs["input_ids"].shape[1]
            pred_text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            target = sample["answer"]
            is_correct = smart_match_answer(pred_text, target, sample.get("answer_aliases"))
            correct += int(is_correct)
            total += 1
            log_entries.append({
                "index": i + 1,
                "question": sample["prompt"],
                "ground_truth": target,
                "original_ground_truth": sample.get("original_answer", target),
                "answer_aliases": sample.get("answer_aliases", []),
                "model_prediction": pred_text,
                "is_correct": is_correct,
            })
            if i < 3:
                status = "CORRECT" if is_correct else "WRONG"
                print(f"    [#{i+1}] GT: {target} | Pred: {pred_text} | {status}")
            del inputs, outputs
            if i % 15 == 0:
                torch.cuda.empty_cache()
                gc.collect()

    acc = correct / total * 100 if total > 0 else 0.0
    print(f"[{name}] Accuracy: {acc:.2f}% ({correct}/{total})")

    name_slug = re.sub(r"[^a-zA-Z0-9_]", "_", name.lower().strip())
    log_path = os.path.join(eval_config["output_dir"], f"eval_details_{name_slug}.json")
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(log_entries, f, ensure_ascii=False, indent=2)
    print(f"   ➔ Đã lưu nhật ký chi tiết tại: {log_path}")
    return acc

train_samples_cv = eval_config["eval_datasets"]["cv_bench"].get("train_samples", 500)
train_samples_mv = eval_config["eval_datasets"]["math_vista"].get("train_samples", 500)
test_samples_cv = eval_config["eval_datasets"]["cv_bench"].get("test_samples", 200)
test_samples_mv = eval_config["eval_datasets"]["math_vista"].get("test_samples", 200)

cv_split_indices = make_stratified_train_test_indices(cv_dataset, train_samples_cv, test_samples_cv, dataset_name="cv_bench", seed=SEED) if cv_dataset is not None else {"train": [], "test": []}
mathvista_split_indices = make_stratified_train_test_indices(mathvista_dataset, train_samples_mv, test_samples_mv, dataset_name="math_vista", seed=SEED) if mathvista_dataset is not None else {"train": [], "test": []}

if cv_dataset is not None:
    print_split_audit(cv_dataset, cv_split_indices, "cv_bench", "CV-Bench")
if mathvista_dataset is not None:
    print_split_audit(mathvista_dataset, mathvista_split_indices, "math_vista", "MathVista")

cv_train = build_formatted_split(cv_dataset, train_samples_cv, "train", dataset_name="cv_bench", indices=cv_split_indices["train"]) if cv_dataset is not None else []
mathvista_train = build_formatted_split(mathvista_dataset, train_samples_mv, "train", dataset_name="math_vista", indices=mathvista_split_indices["train"]) if mathvista_dataset is not None else []
cv_test = build_formatted_split(cv_dataset, test_samples_cv, "test", dataset_name="cv_bench", indices=cv_split_indices["test"]) if cv_dataset is not None else []
mathvista_test = build_formatted_split(mathvista_dataset, test_samples_mv, "test", dataset_name="math_vista", indices=mathvista_split_indices["test"]) if mathvista_dataset is not None else []
combined_train = mathvista_train + cv_train

print(f"Train split: MathVista={len(mathvista_train)}, CV-Bench={len(cv_train)}, combined={len(combined_train)}")
print(f"Test split : MathVista={len(mathvista_test)}, CV-Bench={len(cv_test)}")

print("=" * 60)
print("ZERO-SHOT EVALUATION (BASE MODEL - NO FINE-TUNING)")
print("=" * 60)
restore_initial_trainable_state(model, initial_trainable_state)
zeroshot_results = {}
if cv_test:
    zeroshot_results["cv_bench"] = evaluate_formatted_data(model, cv_test, latent_tokens=None, name="CV-Bench Zero-shot", max_samples=test_samples_cv)
if mathvista_test:
    zeroshot_results["math_vista"] = evaluate_formatted_data(model, mathvista_test, latent_tokens=None, name="MathVista Zero-shot", max_samples=test_samples_mv)

print("\nZero-shot results summary:")
for k, v in zeroshot_results.items():
    print(f"  {k}: {v:.2f}%")


CV-Bench split audit:
  train: n=500 | strata={'3D|Omni3D|Depth': 114, '3D|Omni3D|Distance': 114, '2D|COCO|Count': 84, '2D|COCO|Relation': 68, '2D|ADE20K|Count': 65, '2D|ADE20K|Relation': 55}
  test: n=200 | strata={'3D|Omni3D|Depth': 46, '3D|Omni3D|Distance': 45, '2D|COCO|Count': 34, '2D|COCO|Relation': 27, '2D|ADE20K|Count': 26, '2D|ADE20K|Relation': 22}

MathVista split audit:
  train: n=500 | strata={'multi_choice|text': 270, 'free_form|integer': 209, 'free_form|float': 20, 'free_form|list': 1}
  test: n=200 | strata={'multi_choice|text': 108, 'free_form|integer': 84, 'free_form|float': 8}
Train split: MathVista=500, CV-Bench=500, combined=1000
Test split : MathVista=200, CV-Bench=200
ZERO-SHOT EVALUATION (BASE MODEL - NO FINE-TUNING)
---> Restored 696/696 LoRA tensors + latent rows to initial state.

➔ Đang chạy đánh giá trên CV-Bench Zero-shot (200 mẫu)...


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


    [#1] GT: (C) | Pred: C | CORRECT
    [#2] GT: (A) | Pred: A | CORRECT
    [#3] GT: (A) | Pred: A | CORRECT
[CV-Bench Zero-shot] Accuracy: 68.50% (137/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_cv_bench_zero_shot.json

➔ Đang chạy đánh giá trên MathVista Zero-shot (200 mẫu)...
    [#1] GT: 6 | Pred: 5 | WRONG
    [#2] GT: (C) | Pred: C | CORRECT
    [#3] GT: 1 | Pred: 100 | WRONG
[MathVista Zero-shot] Accuracy: 55.00% (110/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_mathvista_zero_shot.json

Zero-shot results summary:
  cv_bench: 68.50%
  math_vista: 55.00%


## 5b-II. Direct SFT Baseline

Cell này train **Direct SFT** ngay trong notebook, trên đúng `combined_train` gồm 500 MathVista + 500 CV-Bench.

Đặc điểm của Direct SFT:

- Không chèn latent tokens.
- Không dùng bottleneck mask.
- Train 5 epoch, cùng tổng số epoch với LIVR.
- Dùng cùng optimizer, learning rate, grad accumulation và evaluation loop với LIVR.

Đây là baseline quan trọng nhất. Khi đọc kết quả, khoảng cách `LIVR - Direct SFT` mới là tín hiệu chính về việc latent bottleneck có mang lại lợi ích hay không.

In [7]:
# =========================================================================
# CELL 5b-II: DIRECT SFT TRAIN + EVAL TRONG CÙNG NOTEBOOK
# =========================================================================
print("=" * 60)
print("DIRECT SFT TRAINING (NO LATENTS, NO BOTTLENECK)")
print("=" * 60)
restore_initial_trainable_state(model, initial_trainable_state)
model.livr_stage = 2

sft_epochs = eval_config.get("fine_tune_epochs", 5)
direct_sft_results = {}
direct_sft_loss_history = []

if combined_train:
    direct_sft_loss_history = train_model_on_data(
        model,
        combined_train,
        latent_tokens=None,
        epochs=sft_epochs,
        stage1_epochs=0,
        run_name="Direct SFT",
    )

    sft_checkpoint_path = os.path.join(eval_config["output_dir"], "direct_sft_finetuned.pt")
    trainable_sd = extract_lora_state_dict(model)
    torch.save({
        "model_state_dict": trainable_sd,
        "loss_history": direct_sft_loss_history,
        "protocol": {
            "train_samples_per_dataset": 500,
            "test_samples_per_dataset": 200,
            "epochs": sft_epochs,
            "latent_tokens": False,
        },
    }, sft_checkpoint_path)
    print(f"---> Đã lưu Direct SFT checkpoint: {sft_checkpoint_path}")

    print("\n" + "=" * 60)
    print("DIRECT SFT EVALUATION")
    print("=" * 60)
    if cv_test:
        direct_sft_results["cv_bench"] = evaluate_formatted_data(model, cv_test, latent_tokens=None, name="CV-Bench Direct SFT", max_samples=test_samples_cv)
    if mathvista_test:
        direct_sft_results["math_vista"] = evaluate_formatted_data(model, mathvista_test, latent_tokens=None, name="MathVista Direct SFT", max_samples=test_samples_mv)
else:
    raise RuntimeError("combined_train rỗng, không thể train Direct SFT.")

print("\nDirect SFT results summary:")
for k, v in direct_sft_results.items():
    print(f"  {k}: {v:.2f}%")
print(f"Direct SFT loss history: {[round(x, 4) for x in direct_sft_loss_history]}")

DIRECT SFT TRAINING (NO LATENTS, NO BOTTLENECK)
---> Restored 696/696 LoRA tensors + latent rows to initial state.


Direct SFT [Stage 2] Epoch 1/5:  78%|███████▊  | 782/1000 [30:54<08:30,  2.34s/it, Loss=0.6169, Scale=4096]


[WARNING] Direct SFT: NaN/Inf loss tại step 781, bỏ qua batch.


Direct SFT [Stage 2] Epoch 1/5: 100%|██████████| 1000/1000 [40:03<00:00,  2.40s/it, Loss=0.0006, Scale=2048]


➔ Direct SFT Epoch 1/5 - Stage 2 - Average Loss: 0.2748 - Scaler Scale: 2048.0


...


[WARNING] Direct SFT: NaN/Inf loss tại step 781, bỏ qua batch.


Direct SFT [Stage 2] Epoch 5/5:  87%|████████▋ | 874/1000 [34:05<03:34,  1.71s/it, Loss=0.0000, Scale=1024]


[WARNING] Direct SFT: NaN/Inf loss tại step 873, bỏ qua batch.


Direct SFT [Stage 2] Epoch 5/5: 100%|██████████| 1000/1000 [39:16<00:00,  2.36s/it, Loss=0.0000, Scale=1024]


➔ Direct SFT Epoch 5/5 - Stage 2 - Average Loss: 0.0319 - Scaler Scale: 1024.0
---> Đã lưu Direct SFT checkpoint: /kaggle/working/checkpoints/evaluation/direct_sft_finetuned.pt

DIRECT SFT EVALUATION

➔ Đang chạy đánh giá trên CV-Bench Direct SFT (200 mẫu)...
    [#1] GT: (C) | Pred: (C) | CORRECT
    [#2] GT: (A) | Pred: (A) | CORRECT
    [#3] GT: (A) | Pred: (B) | WRONG
[CV-Bench Direct SFT] Accuracy: 80.50% (161/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_cv_bench_direct_sft.json

➔ Đang chạy đánh giá trên MathVista Direct SFT (200 mẫu)...
    [#1] GT: 6 | Pred: 6 | CORRECT
    [#2] GT: (C) | Pred: (C) | CORRECT
    [#3] GT: 1 | Pred: 10 | WRONG
[MathVista Direct SFT] Accuracy: 68.00% (136/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_mathvista_direct_sft.json

Direct SFT results summary:
  cv_bench: 80.50%
  math_vista: 68.00%
Direct SFT loss history: [0.2748, 0.1485, 0.0926, 0.0748, 0.0319]

## 6. Train LIVR Hai Giai Đoạn

Trước khi train LIVR, notebook restore lại LoRA + latent embeddings từ snapshot ban đầu để đảm bảo LIVR và Direct SFT xuất phát từ cùng trạng thái.

LIVR được train theo hai giai đoạn:

| Giai đoạn | Epoch | Attention mask | Mục tiêu |
| :--- | :---: | :--- | :--- |
| Stage 1 | 2 | Bottleneck mask | Ép thông tin ảnh đi qua latent tokens |
| Stage 2 | 3 | Standard causal mask | Học cách kết hợp ảnh gốc và latent tokens khi trả lời |

Notebook in `livr_loss_history` sau khi train. Nếu loss gần như không thay đổi qua các epoch, cần xem đó là lỗi training/gradient thay vì kết quả khoa học hợp lệ.

In [8]:
# =========================================================================
# CELL 6: HUẤN LUYỆN LIVR HAI GIAI ĐOẠN TRÊN CÙNG DATA VỚI DIRECT SFT
# =========================================================================
print("=" * 60)
print("LIVR TRAINING (STAGE 1 BOTTLENECK + STAGE 2 STANDARD MASK)")
print("=" * 60)
restore_initial_trainable_state(model, initial_trainable_state)

livr_stage1_epochs = eval_config.get("livr_stage1_epochs", 2)
livr_stage2_epochs = eval_config.get("livr_stage2_epochs", 3)
livr_epochs = livr_stage1_epochs + livr_stage2_epochs
livr_loss_history = []

if combined_train:
    livr_loss_history = train_model_on_data(
        model,
        combined_train,
        latent_tokens=manager.latent_tokens,
        epochs=livr_epochs,
        stage1_epochs=livr_stage1_epochs,
        run_name="LIVR",
    )

    livr_checkpoint_path = os.path.join(eval_config["output_dir"], "livr_eval_finetuned.pt")
    trainable_sd = extract_lora_state_dict(model)
    torch.save({
        "model_state_dict": trainable_sd,
        "latent_embeddings": model.get_input_embeddings().weight[manager.latent_token_ids].detach().cpu(),
        "loss_history": livr_loss_history,
        "protocol": {
            "train_samples_per_dataset": 500,
            "test_samples_per_dataset": 200,
            "stage1_epochs": livr_stage1_epochs,
            "stage2_epochs": livr_stage2_epochs,
            "latent_tokens": True,
        },
    }, livr_checkpoint_path)
    print(f"---> Đã lưu LIVR checkpoint: {livr_checkpoint_path}")
else:
    raise RuntimeError("combined_train rỗng, không thể train LIVR.")

print(f"LIVR loss history: {[round(x, 4) for x in livr_loss_history]}")
if len(livr_loss_history) >= 2 and max(livr_loss_history) - min(livr_loss_history) < 1e-4:
    print("[WARNING] LIVR loss gần như không đổi. Kiểm tra lại gradient/restore state trước khi tin kết quả.")

LIVR TRAINING (STAGE 1 BOTTLENECK + STAGE 2 STANDARD MASK)
---> Restored 696/696 LoRA tensors + latent rows to initial state.


LIVR [Stage 1] Epoch 1/5:  78%|███████▊  | 782/1000 [31:36<08:38,  2.38s/it, Loss=0.3495, Scale=16384]


[WARNING] LIVR: NaN/Inf loss tại step 781, bỏ qua batch.


LIVR [Stage 1] Epoch 1/5: 100%|██████████| 1000/1000 [40:56<00:00,  2.46s/it, Loss=0.0144, Scale=16384]


➔ LIVR Epoch 1/5 - Stage 1 - Average Loss: 0.4433 - Scaler Scale: 16384.0


...


[WARNING] LIVR: NaN/Inf loss tại step 781, bỏ qua batch.


LIVR [Stage 2] Epoch 5/5:  83%|████████▎ | 827/1000 [32:43<05:43,  1.99s/it, Loss=0.0020, Scale=1024]


[WARNING] LIVR: NaN/Inf loss tại step 826, bỏ qua batch.


LIVR [Stage 2] Epoch 5/5: 100%|██████████| 1000/1000 [39:59<00:00,  2.40s/it, Loss=0.0000, Scale=1024]


➔ LIVR Epoch 5/5 - Stage 2 - Average Loss: 0.0695 - Scaler Scale: 1024.0
---> Đã lưu LIVR checkpoint: /kaggle/working/checkpoints/evaluation/livr_eval_finetuned.pt
LIVR loss history: [0.4433, 0.3372, 0.1973, 0.1225, 0.0695]


In [13]:
# =========================================================================
# HOTFIX: CLEAN OLD STAGE-1 INFERENCE PATCH + CONTIGUOUS SDPA MASK
# =========================================================================
import types
import torch
import torch.nn.functional as F
from src.mask_kaggle import generate_stage1_bottleneck_mask

# 1) Hotfix PyTorch SDPA: đảm bảo attn_mask/bias contiguous ngay trước attention.
if not hasattr(F, "_livr_original_sdpa"):
    F._livr_original_sdpa = F.scaled_dot_product_attention

    def livr_contiguous_sdpa(*args, **kwargs):
        args = list(args)
        if len(args) >= 4 and args[3] is not None:
            args[3] = args[3].contiguous()
        if kwargs.get("attn_mask", None) is not None:
            kwargs["attn_mask"] = kwargs["attn_mask"].contiguous()
        return F._livr_original_sdpa(*args, **kwargs)

    F.scaled_dot_product_attention = livr_contiguous_sdpa
    print("---> Patched SDPA to force contiguous attn_mask.")
else:
    print("---> SDPA contiguous patch already applied.")

# 2) Gỡ các wrapper livr_model_forward cũ.
def get_closure_callable(bound_forward, wanted_name="original_forward"):
    func = getattr(bound_forward, "__func__", bound_forward)
    code = getattr(func, "__code__", None)
    closure = getattr(func, "__closure__", None)
    if code is None or closure is None:
        return None

    for name, cell in zip(code.co_freevars, closure):
        if name == wanted_name:
            try:
                value = cell.cell_contents
            except ValueError:
                return None
            return value if callable(value) else None
    return None

def unwrap_livr_forward(bound_forward):
    current = bound_forward
    depth = 0
    seen = set()

    while True:
        func = getattr(current, "__func__", current)
        func_name = getattr(func, "__name__", "")

        if func_name != "livr_model_forward":
            break

        original = get_closure_callable(current, "original_forward")
        if original is None or id(original) in seen:
            break

        seen.add(id(original))
        current = original
        depth += 1

    return current, depth

base_forward, depth = unwrap_livr_forward(model.model.forward)
model.model.forward = base_forward
print(f"---> Unwrapped {depth} old livr_model_forward wrapper(s).")

# 3) Patch lại inference Stage 1 bản sạch.
def patch_model_for_livr_inference_clean(model, latent_token_ids, image_pad_token_id):
    original_forward = model.model.forward

    def livr_model_forward(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids", args[0] if len(args) > 0 else None)
        stage = getattr(model, "livr_stage", 2)

        if stage == 1 and input_ids is not None:
            kwargs["use_cache"] = False

            batch_size, seq_len = input_ids.size()
            device = input_ids.device
            custom_masks = []

            for b in range(batch_size):
                curr_ids = input_ids[b]
                img_positions = torch.where(curr_ids == image_pad_token_id)[0]

                latent_positions = []
                for lid in latent_token_ids:
                    pos = torch.where(curr_ids == lid)[0]
                    if len(pos) > 0:
                        latent_positions.append(pos[0].item())

                if len(img_positions) == 0 or len(latent_positions) == 0:
                    m_bool = torch.tril(torch.ones(seq_len, seq_len, device=device)).bool()
                    fm = torch.zeros(
                        seq_len,
                        seq_len,
                        dtype=torch.float32,
                        device=device,
                    ).masked_fill(~m_bool, -10000.0)
                else:
                    fm = generate_stage1_bottleneck_mask(
                        seq_len=seq_len,
                        img_start=img_positions[0].item(),
                        img_end=img_positions[-1].item() + 1,
                        prompt_end=min(latent_positions),
                        latent_end=max(latent_positions) + 1,
                        device=device,
                    )

                custom_masks.append(fm.unsqueeze(0))

            kwargs["attention_mask"] = (
                torch.stack(custom_masks, dim=0)
                .to(device=device, dtype=model.dtype)
                .contiguous()
            )

        return original_forward(*args, **kwargs)

    model.model.forward = types.MethodType(livr_model_forward, model.model)
    model._livr_inference_patch_applied = True

    try:
        model.config.use_cache = False
        model.generation_config.use_cache = False
        model.base_model.model.config.use_cache = False
    except Exception as e:
        print(f"[WARNING] Cannot fully disable use_cache: {e}")

    print("---> Clean Stage-1 inference bottleneck patch applied.")

patch_model_for_livr_inference_clean(model, manager.latent_token_ids, manager.image_pad_token_id)
model.livr_stage = 2

---> Patched SDPA to force contiguous attn_mask.
---> Unwrapped 2 old livr_model_forward wrapper(s).
---> Clean Stage-1 inference bottleneck patch applied.


## 7. Đánh Giá LIVR và Sanity Check

Cell này đánh giá model LIVR sau training theo hai chế độ:

| Chế độ | Ý nghĩa |
| :--- | :--- |
| Stage 2 | Inference chính: mô hình thấy ảnh gốc và latent tokens |
| Stage 1 sanity | Áp bottleneck mask khi generate: answer chỉ nhận thông tin ảnh thông qua latent tokens |

Stage 2 accuracy được dùng cho bảng so sánh ba chiều. Stage 1 sanity check giúp kiểm tra latent tokens có thực sự chứa thông tin thị giác hay không.

Cách đọc kết quả:

- Nếu Stage 1 sụt cực mạnh so với Stage 2, latent tokens có thể chưa học được biểu diễn hữu ích.
- Nếu Stage 1 gần Stage 2, đây là tín hiệu tốt cho bottleneck, nhưng vẫn cần đối chiếu với Direct SFT để kết luận LIVR có lợi hơn fine-tuning thường hay không.

In [14]:
# =========================================================================
# CELL 7: ĐÁNH GIÁ LIVR STAGE 2 VÀ STAGE 1 SANITY CHECK
# =========================================================================
print("=== BẮT ĐẦU ĐÁNH GIÁ LIVR ===")
results = {}

if cv_test:
    model.livr_stage = 2
    acc_stage2 = evaluate_formatted_data(
        model, cv_test, latent_tokens=manager.latent_tokens,
        name="CV-Bench LIVR Stage 2", max_samples=test_samples_cv, apply_stage1=False,
    )
    model.livr_stage = 1
    acc_stage1 = evaluate_formatted_data(
        model, cv_test, latent_tokens=manager.latent_tokens,
        name="CV-Bench LIVR Stage 1 Sanity", max_samples=test_samples_cv, apply_stage1=True,
    )
    results["CV-Bench"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1,
    }

if mathvista_test:
    model.livr_stage = 2
    acc_stage2 = evaluate_formatted_data(
        model, mathvista_test, latent_tokens=manager.latent_tokens,
        name="MathVista LIVR Stage 2", max_samples=test_samples_mv, apply_stage1=False,
    )
    model.livr_stage = 1
    acc_stage1 = evaluate_formatted_data(
        model, mathvista_test, latent_tokens=manager.latent_tokens,
        name="MathVista LIVR Stage 1 Sanity", max_samples=test_samples_mv, apply_stage1=True,
    )
    results["MathVista"] = {
        "Stage 2 (Mở)": acc_stage2,
        "Stage 1 (Bịt - Sanity Check)": acc_stage1,
        "Sụt giảm": acc_stage2 - acc_stage1,
    }

if not results:
    raise RuntimeError("Không có test data để đánh giá LIVR.")

print("\n" + "=" * 70)
print(" BẢNG TỔNG KẾT LIVR & SANITY CHECK")
print("=" * 70)
print(f"{'Dataset':<15} | {'Stage 2 (Mở)':<15} | {'Stage 1 (Bịt)':<20} | {'Sụt giảm':<10}")
print("-" * 70)
for ds_name, metrics in results.items():
    print(f"{ds_name:<15} | {metrics['Stage 2 (Mở)']:>13.2f}% | {metrics['Stage 1 (Bịt - Sanity Check)']:>18.2f}% | {metrics['Sụt giảm']:>8.2f}%")
print("=" * 70)

livr_stage2_results = {
    "cv_bench": results.get("CV-Bench", {}).get("Stage 2 (Mở)"),
    "math_vista": results.get("MathVista", {}).get("Stage 2 (Mở)"),
}
print(f"\nlivr_stage2_results = {livr_stage2_results}")
model.livr_stage = 2

=== BẮT ĐẦU ĐÁNH GIÁ LIVR ===

➔ Đang chạy đánh giá trên CV-Bench LIVR Stage 2 (200 mẫu)...
    [#1] GT: (C) | Pred: (C) | CORRECT
    [#2] GT: (A) | Pred: (A) | CORRECT
    [#3] GT: (A) | Pred: (A) | CORRECT
[CV-Bench LIVR Stage 2] Accuracy: 86.00% (172/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_cv_bench_livr_stage_2.json

➔ Đang chạy đánh giá trên CV-Bench LIVR Stage 1 Sanity (200 mẫu)...
    [#1] GT: (C) | Pred: (C) | CORRECT
    [#2] GT: (A) | Pred: (A) | CORRECT
    [#3] GT: (A) | Pred: (B) | WRONG
[CV-Bench LIVR Stage 1 Sanity] Accuracy: 68.50% (137/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle/working/checkpoints/evaluation/eval_details_cv_bench_livr_stage_1_sanity.json

➔ Đang chạy đánh giá trên MathVista LIVR Stage 2 (200 mẫu)...
    [#1] GT: 6 | Pred: 6 | CORRECT
    [#2] GT: (C) | Pred: (C) | CORRECT
    [#3] GT: 1 | Pred: 100 | WRONG
[MathVista LIVR Stage 2] Accuracy: 65.00% (130/200)
   ➔ Đã lưu nhật ký chi tiết tại: /kaggle

## 8. Tổng Kết Ba Chiều

Cell cuối tổng hợp kết quả thành bảng:

`Zero-shot → Direct SFT → LIVR`

Notebook sẽ dừng nếu thiếu bất kỳ biến kết quả nào, thay vì dùng số hardcode. Ngoài bảng in ra màn hình, cell này lưu `three_way_summary.json` vào `output_dir`, gồm:

- Protocol train/test/epoch.
- Accuracy của ba mốc so sánh.
- Loss history của Direct SFT và LIVR.
- Kết quả sanity check Stage 1/Stage 2.

Khi viết báo cáo, nên ưu tiên phân tích `LIVR vs Direct SFT`; Zero-shot chỉ là mốc tham chiếu phụ.

In [15]:
# =========================================================================
# CELL 8: TỔNG KẾT 3-WAY COMPARISON — ZERO-SHOT / DIRECT SFT / LIVR
# =========================================================================
required_vars = ["zeroshot_results", "direct_sft_results", "livr_stage2_results"]
missing = [name for name in required_vars if name not in globals()]
if missing:
    raise RuntimeError(f"Thiếu kết quả {missing}. Hãy chạy đủ các cell trước; notebook không dùng fallback hardcode.")

print("=" * 78)
print("=== BẢNG TỔNG KẾT 3-WAY COMPARISON ===")
print("=" * 78)

datasets = ["cv_bench", "math_vista"]
dataset_names = {"cv_bench": "CV-Bench", "math_vista": "MathVista"}

print(f"{'Dataset':<15} {'Zero-shot':>12} {'Direct SFT':>12} {'LIVR (Ours)':>13} {'LIVR vs SFT':>13}")
print("-" * 78)
for ds in datasets:
    name = dataset_names[ds]
    zs = zeroshot_results.get(ds, float("nan"))
    sft = direct_sft_results.get(ds, float("nan"))
    livr = livr_stage2_results.get(ds, float("nan"))
    delta = livr - sft if (livr == livr and sft == sft) else float("nan")
    delta_str = f"+{delta:.1f}%" if delta > 0 else f"{delta:.1f}%"
    print(f"{name:<15} {zs:>11.1f}% {sft:>11.1f}% {livr:>12.1f}% {delta_str:>13}")
print("-" * 78)

summary_path = os.path.join(eval_config["output_dir"], "three_way_summary.json")
summary_payload = {
    "protocol": {
        "train_samples_per_dataset": 500,
        "test_samples_per_dataset": 200,
        "direct_sft_epochs": eval_config.get("fine_tune_epochs", 5),
        "livr_stage1_epochs": eval_config.get("livr_stage1_epochs", 2),
        "livr_stage2_epochs": eval_config.get("livr_stage2_epochs", 3),
        "split_strategy": "deterministic_stratified_by_dataset_metadata",
        "prompt_policy": "CV-Bench prompt; MathVista query; dataset task prefix; MC answer aliases",
    },
    "zero_shot": zeroshot_results,
    "direct_sft": direct_sft_results,
    "livr_stage2": livr_stage2_results,
    "direct_sft_loss_history": direct_sft_loss_history,
    "livr_loss_history": livr_loss_history,
    "livr_sanity": results,
}
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)
print(f"\nĐã lưu summary tại: {summary_path}")

print("\nDiễn giải nhanh:")
for ds in datasets:
    name = dataset_names[ds]
    zs = zeroshot_results.get(ds, 0.0)
    sft = direct_sft_results.get(ds, 0.0)
    livr = livr_stage2_results.get(ds, 0.0)
    print(f"[{name}] SFT-ZS = {sft - zs:+.1f}%, LIVR-SFT = {livr - sft:+.1f}%")
print("=" * 78)

=== BẢNG TỔNG KẾT 3-WAY COMPARISON ===
Dataset            Zero-shot   Direct SFT   LIVR (Ours)   LIVR vs SFT
------------------------------------------------------------------------------
CV-Bench               68.5%        80.5%         86.0%         +5.5%
MathVista              55.0%        68.0%         65.0%         -3.0%
------------------------------------------------------------------------------

Đã lưu summary tại: /kaggle/working/checkpoints/evaluation/three_way_summary.json

Diễn giải nhanh:
[CV-Bench] SFT-ZS = +12.0%, LIVR-SFT = +5.5%
[MathVista] SFT-ZS = +13.0%, LIVR-SFT = -3.0%
